In [1]:
import pandas as pd
import os

raw = "../data/raw/"
files = sorted([f for f in os.listdir(raw) if f.endswith(".csv")])
print(f"Total CSVs found: {len(files)}\n")

for f in files:
    df = pd.read_csv(raw + f, nrows=3)
    print(f"{f:50s} rows_sample=3  cols={df.shape[1]}")

Total CSVs found: 9

olist_customers_dataset.csv                        rows_sample=3  cols=5
olist_geolocation_dataset.csv                      rows_sample=3  cols=5
olist_order_items_dataset.csv                      rows_sample=3  cols=7
olist_order_payments_dataset.csv                   rows_sample=3  cols=5
olist_order_reviews_dataset.csv                    rows_sample=3  cols=7
olist_orders_dataset.csv                           rows_sample=3  cols=8
olist_products_dataset.csv                         rows_sample=3  cols=9
olist_sellers_dataset.csv                          rows_sample=3  cols=4
product_category_name_translation.csv              rows_sample=3  cols=2


In [2]:
for f in files:
    df = pd.read_csv(raw + f)
    print(f"{f:50s} -> {df.shape[0]:>8,} rows")

olist_customers_dataset.csv                        ->   99,441 rows
olist_geolocation_dataset.csv                      -> 1,000,163 rows
olist_order_items_dataset.csv                      ->  112,650 rows
olist_order_payments_dataset.csv                   ->  103,886 rows
olist_order_reviews_dataset.csv                    ->   99,224 rows
olist_orders_dataset.csv                           ->   99,441 rows
olist_products_dataset.csv                         ->   32,951 rows
olist_sellers_dataset.csv                          ->    3,095 rows
product_category_name_translation.csv              ->       71 rows


In [1]:
import pandas as pd
import os

raw = "../data/raw/"

tables = {
    "customers":   "olist_customers_dataset.csv",
    "orders":      "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "products":    "olist_products_dataset.csv",
    "payments":    "olist_order_payments_dataset.csv",
    "reviews":     "olist_order_reviews_dataset.csv",
    "sellers":     "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

dfs = {}
for name, fname in tables.items():
    dfs[name] = pd.read_csv(raw + fname)
    print(f"\n{'='*70}")
    print(f"TABLE: {name}   shape={dfs[name].shape}")
    print(f"{'='*70}")
    print(dfs[name].dtypes)
    print(dfs[name].head(3))


TABLE: customers   shape=(99441, 5)
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
2                      1151              sao paulo             SP  

TABLE: orders   shape=(99441, 8)
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_

In [2]:
# 1. Is customer_id unique in customers?
print("customers.customer_id unique:", dfs["customers"]["customer_id"].is_unique)
print("orders.order_id unique      :", dfs["orders"]["order_id"].is_unique)
print("products.product_id unique  :", dfs["products"]["product_id"].is_unique)
print("sellers.seller_id unique    :", dfs["sellers"]["seller_id"].is_unique)
print("reviews.review_id unique    :", dfs["reviews"]["review_id"].is_unique)  # may be False!
print("order_items (order_id, order_item_id) unique:",
      not dfs["order_items"].duplicated(subset=["order_id","order_item_id"]).any())

customers.customer_id unique: True
orders.order_id unique      : True
products.product_id unique  : True
sellers.seller_id unique    : True
reviews.review_id unique    : False
order_items (order_id, order_item_id) unique: True


In [3]:
# 2. Every orders.customer_id must exist in customers
orders_cust    = set(dfs["orders"]["customer_id"])
customers_keys = set(dfs["customers"]["customer_id"])
orphans = orders_cust - customers_keys
print(f"orders → customers orphan rows: {len(orphans)}")

# 3. Every order_items.order_id must exist in orders
items_orders = set(dfs["order_items"]["order_id"])
orders_keys  = set(dfs["orders"]["order_id"])
print(f"order_items → orders orphan rows: {len(items_orders - orders_keys)}")

# 4. Every order_items.product_id must exist in products
items_products = set(dfs["order_items"]["product_id"])
prod_keys      = set(dfs["products"]["product_id"])
print(f"order_items → products orphan rows: {len(items_products - prod_keys)}")

# 5. Every order_items.seller_id must exist in sellers
items_sellers = set(dfs["order_items"]["seller_id"])
seller_keys   = set(dfs["sellers"]["seller_id"])
print(f"order_items → sellers orphan rows: {len(items_sellers - seller_keys)}")

# 6. Every payments.order_id must exist in orders
pay_orders = set(dfs["payments"]["order_id"])
print(f"payments → orders orphan rows: {len(pay_orders - orders_keys)}")

# 7. Every reviews.order_id must exist in orders
rev_orders = set(dfs["reviews"]["order_id"])
print(f"reviews → orders orphan rows: {len(rev_orders - orders_keys)}")

orders → customers orphan rows: 0
order_items → orders orphan rows: 0
order_items → products orphan rows: 0
order_items → sellers orphan rows: 0
payments → orders orphan rows: 0
reviews → orders orphan rows: 0


In [ ]:
# 8. Orders count vs orders+items joined count
n_orders       = dfs["orders"]["order_id"].nunique()
n_joined_rows  = dfs["orders"].merge(dfs["order_items"], on="order_id").shape[0]

print(f"Unique orders                         : {n_orders:,}")
print(f"Rows after orders ⋈ order_items       : {n_joined_rows:,}")
print(f"Ratio (avg items per order)           : {n_joined_rows/n_orders:.2f}")
print(f"Orders with >1 item                    : "
      f"{(dfs['order_items'].groupby('order_id').size() > 1).sum():,}")